# FP_NB_SchemaDrift_Test
**Northbrook Lending: what happens when the source schema changes?**

**Test scenario (a real change, not a simulation)**
1. In the source SQL database, run `ALTER TABLE dbo.Branch ADD BranchManager nvarchar(100) NULL` and populate it (script `sql/04_schema_drift_change.sql`).
2. Fabric mirroring replicates the change to OneLake, so the Bronze shortcut shows the new column with no action needed.
3. This notebook **detects** drift by comparing Bronze and Silver schemas for every table, **classifies** each difference, **runs the Silver transform**, then **re-checks**.
4. Every result is logged to `FP_Gold.dbo.SchemaDrift_Log`.

**Policy demonstrated**
| Change | Classification | Handling |
|---|---|---|
| New column | Additive – non-breaking | Flows to Silver automatically (`overwriteSchema` / `mergeSchema`) |
| Column removed / renamed | Breaking | Logged for manual review; Silver keeps the existing column |
| Data type changed | Breaking | Logged for manual review |

Gold is a **contract layer**: its tables select explicit columns, so a new source column does not reach reports until it is deliberately added to the star schema and semantic model.


In [ ]:
# Parameter - the pipeline passes its run ID here; it stays empty when you run the notebook by hand
run_id = ""

In [ ]:
# ---------- Setup ----------
# uuid for run IDs, json for the exit value, datetime for timestamps
import uuid, json, datetime as dt
# Spark SQL functions such as col and lit - imported as F
from pyspark.sql import functions as F

# Use the pipeline run ID if one was passed in, otherwise make a random one for manual runs
run_id = run_id or str(uuid.uuid4())

# Find the OneLake root folder of a lakehouse by name, in whichever workspace this runs
def lakehouse_root(name: str) -> str:
    try:
        # Preferred - ask Fabric for this lakehouse by name
        lh = notebookutils.lakehouse.get(name)
        # The lookup can return a dict or an object - read its properties either way
        props = lh["properties"] if isinstance(lh, dict) else lh.properties
        # Take its OneLake Tables path and cut off /Tables to leave the lakehouse root
        return props["oneLakeTablesPath"].rstrip("/").rsplit("/Tables", 1)[0]
    except Exception:
        # Fallback if the lookup fails - build the OneLake path from the current workspace name
        ws = notebookutils.runtime.context["currentWorkspaceName"]
        return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{name}.Lakehouse"

# Build the full path to a Delta table - root/Tables/dbo/TableName
def table_path(root, table, schema="dbo"):
    return f"{root}/Tables/{schema}/{table}"

# Find the Bronze, Silver and Gold lakehouse roots
BRONZE, SILVER, GOLD = (lakehouse_root(x) for x in ("FP_Bronze", "FP_Silver", "FP_Gold"))
# Tables whose schemas are compared
TABLES = ["Branch", "Product", "Customer", "LoanApplication", "Loan", "Repayment", "Arrears"]
# Columns that exist only in Silver, so they are not reported as drift
SILVER_ONLY = {"CustomerVersion", "ValidFrom", "ValidTo", "IsCurrent"}       # SCD2 columns added in Silver

# Read a table's columns as {name: type}, ignoring the underscore audit columns
def schema_of(path):
    return {f.name: f.dataType.simpleString() for f in spark.read.format("delta").load(path).schema.fields
            if not f.name.startswith("_")}

# The type a Bronze column should have in Silver, after Silver's standard type changes
def expected_silver_type(bronze_type):
    """Silver's deliberate type conformance (see FP_NB_Silver_Transform)."""
    # Small whole numbers are widened to int
    if bronze_type in ("smallint", "tinyint"):
        return "int"
    # Decimals are standardised to decimal(18,2)
    if bronze_type.startswith("decimal"):
        return "decimal(18,2)"
    # Any other type should be unchanged
    return bronze_type

# Compare every table's Bronze and Silver schemas and list any differences, labelled with the test phase
def compare(phase):
    rows = []
    for t in TABLES:
        # Read the column list of the table in Bronze and in Silver
        b, s = schema_of(table_path(BRONZE, t)), schema_of(table_path(SILVER, t))
        # Columns in Bronze that Silver doesn't have yet
        added = sorted(set(b) - set(s))
        # Columns in Silver that are gone from Bronze, ignoring the Silver-only SCD2 columns
        removed = sorted(set(s) - set(b) - SILVER_ONLY)
        # Columns in both whose Silver type is not what the Bronze type should become
        retyped = sorted(f"{c}: {s[c]} -> {b[c]}" for c in set(b) & set(s) if expected_silver_type(b[c]) != s[c])
        # Classify - removed or retyped columns are Breaking; new columns only are Additive; otherwise No drift
        if removed or retyped:
            cls, action = "Breaking", "Manual review required – load continues, existing Silver columns kept"
        elif added:
            cls, action = "Additive (non-breaking)", "Auto-evolve Silver schema on next load"
        else:
            cls, action = "No drift", "None"
        # One result row per table, with the differences joined into text
        rows.append((run_id, dt.datetime.utcnow(), phase, t, ", ".join(added), ", ".join(removed), "; ".join(retyped), cls, action))
    return rows

# Column names and types for the drift results table
cols = "RunId string, CheckedAt timestamp, Phase string, TableName string, ColumnsAdded string, ColumnsRemoved string, TypeChanges string, Classification string, Action string"

In [ ]:
# ---------- Step 1: detect drift BEFORE the pipeline runs ----------
# Compare Bronze with Silver as things stand now
before = compare("Before Silver load")
# Show the differences found for each table
display(spark.createDataFrame(before, cols).select("TableName", "ColumnsAdded", "ColumnsRemoved", "TypeChanges", "Classification", "Action"))
# List the tables whose classification is anything other than No drift
drifted = [r[3] for r in before if r[7] != "No drift"]
print("Tables with drift:", drifted or "none")

In [ ]:
# ---------- Step 2: run the normal Silver transform (exactly what the daily pipeline does) ----------
# Run FP_NB_Silver_Transform with this run ID and a 30-minute (1,800-second) timeout; it rewrites the Silver tables
silver_result = notebookutils.notebook.run("FP_NB_Silver_Transform", 1800, {"run_id": run_id})
# Show the Silver notebook's exit summary
print(silver_result)

In [ ]:
# ---------- Step 3: re-check, show the evolved Silver table, and log everything ----------
# Compare Bronze with Silver again now the load has run
after = compare("After Silver load")
# Show the comparison after the load
display(spark.createDataFrame(after, cols).select("TableName", "ColumnsAdded", "ColumnsRemoved", "TypeChanges", "Classification"))

# For each table that drifted, show its new columns now present in Silver
for t in drifted:
    # Look up the new column names found for this table in step 1
    new_cols = next(r[4] for r in before if r[3] == t).split(", ")
    print(f"\n{t}: new column(s) now in Silver ->", new_cols)
    # Show up to 12 Silver rows - the first two ID or Name columns plus the new columns
    display(spark.read.format("delta").load(table_path(SILVER, t)).select(*[c for c in spark.read.format("delta")
            .load(table_path(SILVER, t)).columns if c.endswith("ID") or c.endswith("Name")][:2], *new_cols).limit(12))

# Append the before and after results to SchemaDrift_Log in Gold
log = spark.createDataFrame(before + after, cols)
log.write.format("delta").mode("append").save(table_path(GOLD, "SchemaDrift_Log"))

# Tables that drifted before the load and show No drift after it
resolved = [r[3] for r in after if r[7] == "No drift" and r[3] in drifted]
# Summary - tables that drifted, which were fixed by the load, and which still need attention
result = {"notebook": "FP_NB_SchemaDrift_Test", "run_id": run_id, "drift_detected": drifted,
          "auto_resolved": resolved, "still_open": [t for t in drifted if t not in resolved]}
# Print the test summary
print(json.dumps(result, indent=2))

In [ ]:
# End the notebook and return the test summary to whatever called it
notebookutils.notebook.exit(json.dumps(result))